# HADL – full evaluation pipeline on Google Colab (A100 / L4 / T4)

Runs steps 0–7 of `evaluation_pipeline/` end to end from the **`main`** branch of `pratyushk22/hadl_courseproject`:
re-profile the time proxy on this GPU → costed candidate pool → subset selection → Qwen3-1.7B + LoRA training
→ evaluation → covariance-shift diagnostic → tables → figures.

**Colab:** Runtime → Change runtime type → A100 (or L4/T4). For Kaggle use `run_pipeline_kaggle.ipynb` instead.

In [ ]:
# 1) Get the code
REPO_URL = "https://github.com/pratyushk22/hadl_courseproject.git"
BRANCH = "main"
!git clone --depth 1 -b $BRANCH $REPO_URL hadl_courseproject
%cd hadl_courseproject/evaluation_pipeline
import os
assert os.path.isfile("run_pipeline.sh"), "this branch has no evaluation_pipeline/ yet - merge it first or change REPO_URL/BRANCH"
!nvidia-smi

In [ ]:
# 2) Dependencies (torch is pre-installed on Colab); versions tested with this pipeline
!pip -q install "transformers==4.57.*" "peft==0.22.*" "accelerate>=0.34" "datasets>=2.19" "jinja2>=3.1" gdown pyarrow scikit-learn scipy matplotlib pytest

In [ ]:
# 3) Quick sanity check: unit tests (CPU, ~10 s)
!python -m pytest -q tests -p no:cacheprovider

### 4) Run the whole pipeline
* `PROFILE=1` re-fits the training-time proxy on *this* GPU (the committed proxy was profiled on Apple MPS), ~5 min.
* Budget = `BUDGET_X_REAL` × proxy time of the real train split (1.0 ≈ “as much synthetic training time as real”).
* On a 16 GB GPU (T4/L4) add `BATCH=2 ACCUM=8 EVAL_BS=4` (same effective batch size).
* Step 7 downloads the embedding files from the team Google Drive and runs the covariance diagnostic automatically.
* Finished runs are skipped, so re-running after a disconnect resumes where it stopped (same session only).

In [ ]:
!PROFILE=1 SEEDS="42" BUDGET_X_REAL="1.0" METHODS="random quality_only proposed_knapsack" REFERENCE="real_only" bash run_pipeline.sh

Optional, more thorough (≈3× the GPU time): ablations + 3 seeds + 3 budgets

```
!SEEDS="42 43 44" BUDGET_X_REAL="0.5 1 2" METHODS=all REFERENCE=real_only STEPS="1 2 3 4 5 6 7" bash run_pipeline.sh
```

In [ ]:
# 5) Look at the results
import json, glob, os
from IPython.display import Markdown, Image, display
for f in ["results_table.md", "significance_vs_random.md", "covariance_diagnostic.md"]:
    p = f"outputs/results/{f}"
    if os.path.exists(p):
        display(Markdown(f"### {f}\n" + open(p).read()))
for name in ["qa_scores_test", "quality_vs_training_time_test", "training_time_predicted_vs_actual",
             "proxy_step_parity", "pipeline_overhead", "training_loss_curves"]:
    p = f"outputs/figures/{name}.png"
    if os.path.exists(p):
        display(Image(p, width=900))
for p in sorted(glob.glob("outputs/figures/selection_overview_b*.png") + glob.glob("outputs/figures/covariance_shift_b*.png")):
    display(Image(p, width=900))

In [ ]:
# 6) Download everything (tables, figures, predictions, adapters, logs)
!zip -qr hadl_results.zip outputs -x "outputs/pool/*"
from google.colab import files
files.download("hadl_results.zip")